**Jissy Jayaprakash**



In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Set your working path
data_path = "/content/preprocessed_news.csv"
output_path = "/content/news_with_bias_and_fake_labels.csv"



Mounted at /content/drive


In [2]:
!pip install transformers pandas tqdm


In [3]:
from transformers import pipeline
import pandas as pd
from tqdm import tqdm

# Load classifier (runs on GPU if available)
zero_shot_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)

# Define labels
labels = ["left-wing", "right-wing", "neutral", "real", "fake"]
bias_set = {"left-wing", "right-wing", "neutral"}
truth_set = {"real", "fake"}


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [4]:
def detect_bias_and_truth(text):
    try:
        result = zero_shot_classifier(text, candidate_labels=labels)
        scores = dict(zip(result["labels"], result["scores"]))

        truth_label = max(truth_set, key=lambda l: scores.get(l, 0))
        truth_confidence = scores[truth_label]

        bias_label = max(bias_set, key=lambda l: scores.get(l, 0))
        bias_confidence = scores[bias_label]

        return truth_label, truth_confidence, bias_label, bias_confidence
    except:
        return "error", 0.0, "error", 0.0


In [5]:
df = pd.read_csv(data_path, encoding='ISO-8859-1')

tqdm.pandas()

# Use only first 512 characters for speed (optional)
df["text_for_analysis"] = df["translated_text"].astype(str).str.slice(0, 512)

# Run classification
df[["truth_prediction", "truth_confidence", "bias_label", "bias_confidence"]] = df["text_for_analysis"].progress_apply(
    lambda x: pd.Series(detect_bias_and_truth(x))
)

# Save output
df.to_csv(output_path, index=False)
print("✅ Done! Output saved to:", output_path)


100%|██████████| 15298/15298 [36:43<00:00,  6.94it/s]


✅ Done! Output saved to: /content/news_with_bias_and_fake_labels.csv


In [6]:
from google.colab import files
files.download("news_with_bias_and_fake_labels.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
from google.colab import files
import zipfile

# Upload the dataset
uploaded = files.upload()  # select liar_dataset.zip from your computer

# Unzip
with zipfile.ZipFile("liar_dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("liar_dataset")


Saving liar_dataset.zip to liar_dataset.zip


In [8]:
import pandas as pd

# Load train.tsv
df_liar = pd.read_csv("liar_dataset/train.tsv", sep="\t", header=None, names=[
    "id","label","statement","subject","speaker","job_title","state_info",
    "party_affiliation","barely_true_counts","false_counts","half_true_counts",
    "mostly_true_counts","pants_on_fire_counts","context"
])

# Map to binary real/fake
def truth_label(label):
    return "fake" if label in ["false", "pants-fire"] else "real"

df_liar["truth_label_gt"] = df_liar["label"].apply(truth_label)
df_liar = df_liar.rename(columns={"statement": "translated_text"})
df_liar["text_for_analysis"] = df_liar["translated_text"].astype(str).str.slice(0, 512)


In [9]:
from tqdm import tqdm
tqdm.pandas()

df_liar[["truth_prediction", "truth_confidence", "bias_label", "bias_confidence"]] = df_liar["text_for_analysis"].progress_apply(
    lambda x: pd.Series(detect_bias_and_truth(x))
)


100%|██████████| 10240/10240 [16:48<00:00, 10.15it/s]


In [10]:
from sklearn.metrics import classification_report

print("🧪 Fake News Detection Performance (vs LIAR ground truth):")
print(classification_report(df_liar["truth_label_gt"], df_liar["truth_prediction"]))


🧪 Fake News Detection Performance (vs LIAR ground truth):
              precision    recall  f1-score   support

        fake       0.47      0.05      0.09      2834
        real       0.73      0.98      0.84      7406

    accuracy                           0.72     10240
   macro avg       0.60      0.51      0.46     10240
weighted avg       0.66      0.72      0.63     10240



In [12]:
zero_shot_classifier.device = -1  # force CPU mode

import pickle
with open("bias_fake_model.pkl", "wb") as f:
    pickle.dump(zero_shot_classifier, f)

print("✅ Pickle file saved as CPU-compatible 'bias_fake_model.pkl'")


✅ Pickle file saved as CPU-compatible 'bias_fake_model.pkl'


In [14]:
from google.colab import files
files.download("bias_fake_model.pkl")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>